# Inhoudsopgave
## Documentatie / uitleg
- Productoverzicht
- Stakeholder analyse
- Datavereisten
- Modelvereisten
- Onderhoud en hertraining
- Data pipeline
- Modellering
- Deployment
- CI/CD
- Monitoring
## Technisch onderdeel
- Data loading
- Data preprocessing and feature engineering
- Model training
- Deployment
- Monitoring

# Documentatie & uitleg

Hieronder wordt uitleg gegeven over de inhoud van het notebook.
### Productoverzicht:
Dit project maakt een intelligent retail analytics systeem voor BrightMart, een middelgrote retailer. Het doel van het systeem is om winkels efficiënter te maken door edge-based klantdetectie te combineren met cloudgebaseerde vraagvoorspellingen.

Het systeem bestaat uit twee modellen:
- Een cloud based model dat de vraag naar producten voorspelt op basis van historische verkoopgegevens.
- Een edge model dat het aantal klanten in een winkel inschat op basis van camerabeelden

Met dit systeem kunnen er real-time inzichten verkregen worden voor store managers en kan de voorraad beter bijgehouden worden voor het supply chain team.

### Stakeholders:
- Store managers: Hebben behoefte aan real time inzicht van de winkel bezetting en de omzet van een winkel.
- Supply chain team: Hebben behoefte aan een nauwkeurige voorspelling van verkoop.
- IT-afdeling: Wilt een data pipeline waar niet veel aan gedaan hoeft te worden.

### Datavereisten:
Het systeem maakt gebruik van twee soorten data:
- Retaildata: Deze data bevat informatie over winkel, product, datum en verkoop.
- Beelddata: Deze wordt gebruikt om klanten te detecteren en te tellen.

De eisen van de data zijn als volgt:
- Kwaliteit: De data moet schoon zijn. (Missende waardes worden verwijderd)
- Volume: De pipeline is schaalbaar en kan volume aan.
- Snelheid: Het edge model moet real-time voorspellingen kunnen maken.
- Privacy: De beelddata wordt lokaal verwerkt en niet opgeslagen.
- Veiligheid: Data wordt veilig opgeslagen en verwerkt binnen een beveiligd platform
- Vorm: De beelddata wordt in .npy bestanden aangeleverd.

### Model vereisten
Er worden twee modellen gebruikt:

#### Cloud model:
- Taak: Voorspellen van het aan verkochte items van aankomende dagen zodat het supply chain team spullen kan inkopen.
- Type model: RandomForestRegressor (SparkML)
- Eisen:
  - Zo laag mogelijke RMSE
  - Schaalbaar
  - MLflow integratie voor tracking en versiebeheer

#### Edge model:
- Taak: Het aantal klanten in een winkel voorspellen aan de hand van camerabeelden.
- Type model: Regressie (Lineaire regressie of RandomForest)
- Eisen:
  - Real time voorspellingen kunnen maken
  - Zo laag mogelijke RMSE

#### Onderhoud & hertraining:
Het systeem wordt ontworpen om zich aan te passen aan verandering.
- Modelprestaties worden gemonitord aan de hand van de RMSE.
- Data drift wordt gedetecteerd, bij data drift krijgt gebruiker een melding om hertraining in te plannen.
- Data drift detectie kan per model worden aangepast naar andere hoeveelheid.

#### Data pipeline:
De data pipeline bestaat uit:
- Data ingestion: Het inladen van de retail- en beelddata.
- Data cleaning: Het verwijderen van ongeldige en dubbele waardes
- Feature engineering: Het toevoegen van features zoals datum componenten en lag-variabelen.
- Data splitsen: Het maken van train/test sets.

#### Modellering:
De modelleringspipeline bevat:
- Feature engineering (VectorAssembler voor SparkML)
- Model training (RandomForest en Lineaire regressie)
- Evaluatie met RMSE
- Experiment tracking met behulp van MLFlow

#### Deployment:
De modellen worden geladen vanuit MLFlow en gebruikt voor voorspellingen.

#### CI/CD:
Het systeem ondersteund het volgende:
- Continuous Integration: Modellen worden bijgehouden in MLFlow.
- Continuous Deployment: Nieuwe modellen kunnen toegepast worden.

#### Monitoring:
Modelprestaties worden gemonitord:
- RMSE voor evaluatie.
- Drift detection vergelijkt voorspellingen met echte waardes.
- Bij afwijkingen krijgt gebruiker een melding om modellen opnieuw te trainen.

# Technisch onderdeel

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor as SparkRF
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.window import Window
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator


from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor as SKLearnRF
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D,MaxPooling2D,Flatten,Dense)
from mlflow.models import infer_signature

from PIL import Image
import mlflow
import os
import numpy as np
import pandas as pd
from abc import ABC, abstractmethod
from scipy.stats import ks_2samp

# Data loading

In [0]:
store_demand_schema = StructType([
  StructField('date', DateType(), False),
  StructField('store', IntegerType(), False),
  StructField('item', IntegerType(), False),
  StructField('sales', IntegerType(), False)
])
ecom_schema = StructType([
  StructField('InvoiceNo', IntegerType(), False),
  StructField('StockCode', IntegerType(), False),
  StructField('Description', StringType(), False),
  StructField('Quantity', IntegerType(), False),
  StructField('InvoiceDate', StringType(), False),
  StructField('UnitPrice', FloatType(), False),
  StructField('CustomerID', IntegerType(), False),
  StructField('Country', StringType(), False)
])

base_path = "/Volumes/workspace/default/course_files/"

ecom_df = spark.read.csv(
    base_path + "ecom/data.csv",
    header=True,
    schema=ecom_schema
)

retail_train = spark.read.csv(
    base_path + "retail/train.csv",
    header=True,
    schema=store_demand_schema
)

retail_test = spark.read.csv(
    base_path + "retail/test.csv",
    header=True,
    schema=store_demand_schema
)

images_df = spark.read.format("binaryFile").load(
    base_path + "person_detection_small/images/"
).select("path")

images = np.load(base_path + "surv_camera/images_400.npy")
labels = np.load(base_path + "surv_camera/labels_400.npy")

print("Ecom rows:", ecom_df.count())
print("Retail train rows:", retail_train.count())

print("Images numpy shape:", images.shape)
print("Labels numpy shape:", labels.shape)

display(ecom_df)
display(retail_train)
display(images_df.limit(5))

# Data preprocessing and feature engineering

Hieronder worden functies aangemaakt om de data op te schonen en extra features aan te maken. Dit wordt gedaan middels zelf-gedefineerde functies zodat er modulair gewerkt kan worden en het makkelijk is om dingen aan te passen of te veranderen. 

Daarna worden de dataframes bewerkt door middel van de functies en wordt het dataframe klaargemaakt om gebruikt te worden in een machine-learning model.

In [0]:
def clean_ecom(df):
    return (
        df
        .filter(col("Quantity") > 0)
        .filter(col("UnitPrice") > 0) #Bij validatie kwam naar voren dat er ~2500 unitsprices waren die minder dan 0 UnitPrice waren.
        .filter(col("StockCode").isNotNull())
        .dropDuplicates()
    )

def clean_retail(df):
    return df \
        .filter(col("sales") > 0) \
        .dropDuplicates() \
        .dropna()

def create_retail_features(df):
    return (
        df
        .withColumn("year", year("date"))
        .withColumn("month", month("date"))
        .withColumn("day", dayofmonth("date"))
        .withColumn("full_date", col("date"))
    )

def create_ecom_features(df):
    return (
        df
        .withColumn(
            "invoice_timestamp",
            to_timestamp(
                "InvoiceDate",
                "M/d/yyyy H:mm"
            )
        )
        .withColumn(
            "revenue",
            col("Quantity") * col("UnitPrice")
        )
        .withColumn(
            "invoice_date",
            to_date(col("invoice_timestamp"))
        )
        .withColumn(
            "year",
            year("invoice_timestamp")
        )
        .withColumn(
            "month",
            month("invoice_timestamp")
        )
        .withColumn(
            "day",
            dayofmonth("invoice_timestamp")
        )
        .withColumn(
            "day_of_week",
            dayofweek("invoice_timestamp")
        )
    )


def create_retail_gold(df):
    return df.groupBy("store", "item", "year", "month", "day","full_date").agg(
    sum("sales").alias("sales")
    )

In [0]:
def validate_retail_df(df):

    print(f"Rows: {df.count()}")

    print("\nSchema:")
    df.printSchema()

    # Null checks
    assert df.filter(col("date").isNull()).count() == 0, \
        "Null values found in date"

    assert df.filter(col("store").isNull()).count() == 0, \
        "Null values found in store"

    assert df.filter(col("item").isNull()).count() == 0, \
        "Null values found in item"

    assert df.filter(col("sales").isNull()).count() == 0, \
        "Null values found in sales"

    # Business rules
    assert df.filter(col("sales") < 0).count() == 0, \
        "Negative sales detected"

    assert df.filter(
        (col("month") < 1) | (col("month") > 12)
    ).count() == 0, \
        "Invalid month detected"

    assert df.filter(
        (col("day") < 1) | (col("day") > 31)
    ).count() == 0, \
        "Invalid day detected"

    print("\nRetail validation passed")


def validate_ecom_df(df):

    assert df.filter(
        col("InvoiceDate").isNull()
    ).count() == 0, \
        "Null InvoiceDate values found"

    assert df.filter(
        col("Quantity") <= 0
    ).count() == 0, \
        "Invalid Quantity values found"

    assert df.filter(
        col("UnitPrice") <= 0
    ).count() == 0, \
        "Invalid UnitPrice values found"

    print("E-commerce validation passed")

class MLPipeline(ABC):

    def apply(self, df):

        df_clean = self._clean(df)

        df_features = self._features(df_clean)

        self._validate(df_features)

        return df_features

    @abstractmethod
    def _clean(self, df):
        pass

    @abstractmethod
    def _features(self, df):
        pass

    @abstractmethod
    def _validate(self, df):
        pass

class RetailPipeline(MLPipeline):

    def _clean(self, df):
        return clean_retail(df)

    def _features(self, df):
        return create_retail_features(df)

    def _validate(self, df):
        validate_retail_df(df)

class EcomPipeline(MLPipeline):

    def _clean(self, df):
        return clean_ecom(df)

    def _features(self, df):
        return create_ecom_features(df)

    def _validate(self, df):
        validate_ecom_df(df)

In [0]:
retail_pipeline = RetailPipeline()
retail_features = retail_pipeline.apply(retail_train)

ecom_pipeline = EcomPipeline()
ecom_features = ecom_pipeline.apply(ecom_df)

retail_gold = create_retail_gold(retail_features)

# Lag-features toevoegen voor forecasting
window = Window.partitionBy(
    "store",
    "item"
).orderBy(
    "year",
    "month",
    "day"
)

retail_gold = (
    retail_gold
    .withColumn(
        "lag_1",
        lag("sales", 1).over(window)
    )
    .withColumn(
        "lag_7",
        lag("sales", 7).over(window)
    )
    .dropna()
)

ecom_gold = (
    ecom_features
    .groupBy(
        "year",
        "month"
    )
    .agg(
        sum("revenue").alias("monthly_revenue")
    )
    .orderBy(
        "year",
        "month"
    )
)

In [0]:
ecom_df.filter(col("UnitPrice") <= 0).count()

# Schaalbaarheid

Om de schaalbaarheid aan te tonen gaan we de dataset vergroten en door de pipeline heen halen. Er is te zien dat het langer duurt naarmate de dataset wordt vergroot maar dit zijn geen extreem grote stappen zijn, dus is de pipeline schaalbaar.

In [0]:
import time

results = []

base_df = [retail_train, ecom_df]
for df in base_df:
    for multiplier in [1, 2, 4, 8]:

        test_df = df

        current = 1

        while current < multiplier:
            test_df = test_df.unionByName(test_df)
            current *= 2

        rows = test_df.count()

        start = time.time()

        if df == ecom_df:
            features = ecom_pipeline.apply(test_df)
        else:   
            features = retail_pipeline.apply(test_df)

        features.count()

        runtime = time.time() - start

        results.append(
            (multiplier, rows, runtime)
        )

for multiplier, rows, runtime in results:

    print(
        f"Multiplier={multiplier} | "
        f"Rows={rows:,} | "
        f"Runtime={runtime:.2f}s | "
    )

# Model training

## Cloud model training
Hieronder wordt er een cloud model gemaakt om een voorspelling te maken van het aantal "items" wat verkocht gaat worden. Dit wordt aan de hand van een RandomForestRegressor model gedaan omdat deze een aantal voordelen heeft:
  - Robuust tegen ruis
  - Kan niet lineaire verbanden ontdekken
  - Heeft geen ingewikkelde pre-processing nodig\
Hierdoor is RandomForestRegressor een goed baseline model.

Er wordt een assembler gebruikt die alle features omzet naar één vector omdat spark modellen zo werken.
Verder wordt de laatste 20% als test gebruikt, dit omdat de data tijdsgebonden is.

Er worden twee modellen getraind, een op retail data en een op ecom data. 

## Retail data

Hieronder wordt op de retail data een SparkRF model getraind, omdat spark een Vector als input nodig heeft, worden alle features omgezet naar een vector middels een VectorAssembles, deze maakt een nieuwe kolom met de FeatureVector. Hierna wordt het model gedefineerd met de feature kolom en het label kolom. Daarna worden beide gecombineerd en in een pipeline gestopt.

De data is gesplits op ongeveer de laatste 20% van de dataset omdat we te maken hebben met sequentiele data. Hierdoor kan er dus niet zomaar gehusseld worden. Middels de .filter methode worden de train en test set aangemaakt.

Voor de evaluatie wordt de Root Mean Squared Error (RMSE) gebruikt. Deze metriek geeft aan hoe ver de voorspellingen gemiddeld afwijken van de werkelijke verkoopcijfers, waarbij lagere waarden betere prestaties betekenen.

Om de optimale hyperparameters te vinden wordt gebruikgemaakt van een CrossValidator. Hierbij worden verschillende combinaties van de parameters maxDepth en numTrees geëvalueerd. De parameter maxDepth bepaalt hoe diep de beslisbomen mogen groeien, terwijl numTrees aangeeft uit hoeveel bomen het Random Forest bestaat. Door middel van 3-fold cross-validatie wordt voor iedere parametercombinatie de gemiddelde prestatie bepaald, waarna automatisch het best presterende model wordt geselecteerd.

Nadat het model is getraind worden voorspellingen gemaakt op de testset en wordt de uiteindelijke RMSE berekend. Daarnaast worden de beste hyperparameters vastgelegd. Ten slotte wordt het model samen met de prestatiecijfers gelogd in MLflow. Hierdoor kunnen experimenten eenvoudig worden vergeleken, gereproduceerd.

In [0]:
# Assembler aanmaken
assembler = VectorAssembler(
    inputCols=["store", "item", "year", "month", "day", "lag_1", "lag_7"],
    outputCol="features"
)

# RandomForest model aanmaken
rf = SparkRF(
    featuresCol="features",
    labelCol="sales"
)
# Pipeline bouwen
pipeline = Pipeline(stages=[assembler, rf])

In [0]:
#Ongeveer laatste 20% van de tijd gebruiken als testset
split_date = "2017-01-01"

retail_train = retail_gold.filter(
    col("full_date") < split_date
)

retail_test = retail_gold.filter(
    col("full_date") >= split_date
)

print("Train rows:", retail_train.count())
print("Test rows:", retail_test.count())

In [0]:
print("Training period:")
retail_train.selectExpr("min(full_date)","max(full_date)").show()

print("Testing period:")
retail_test.selectExpr("min(full_date)","max(full_date)").show()

# Evaluator aanmaken met RMSE als metric
retail_evaluator = RegressionEvaluator(
    labelCol="sales",
    predictionCol="prediction",
    metricName="rmse"
)

paramGrid = (
    ParamGridBuilder()
    .addGrid(rf.maxDepth, [5, 10])
    .addGrid(rf.numTrees, [20, 50])
    .build()
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=retail_evaluator,
    numFolds=3
)

In [0]:
# Tijdelijke opslaglocatie voor modellen aanmaken
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/course_files/tmp/"
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/course_files/tmp/"

with mlflow.start_run(run_name="retail_forecast"):

    cv_model = cv.fit(retail_train)

    preds = cv_model.transform(retail_test)

    rmse = retail_evaluator.evaluate(preds)

    best_rf = cv_model.bestModel.stages[-1]

In [0]:
mlflow.log_metric("rmse", rmse)

mlflow.log_param("best_maxDepth",best_rf.getMaxDepth())
mlflow.log_param("best_numTrees",best_rf.getNumTrees)
mlflow.spark.log_model(cv_model.bestModel,"best_retail_model")

print(f"Best RMSE: {rmse}")
print(f"Best depth: {best_rf.getMaxDepth()}")
print(f"Best trees: {best_rf.getNumTrees}")

## E-com data

Omdat er tijdens het serven van het RF model van Spark problemen waren, is het model omgezet naar een SKLearn RF model. Hierdoor wordt de data eerst omgezet naar pandas, om het als input te kunnen gebruiken, verder worden de hyperparameters gebruikt die bij het SparkRF model het beste presteerde. Vervolgens wordt het model getraind en gelogd.

https://dbc-f91f8233-2c2e.cloud.databricks.com/ml/endpoints/ecommerce-forecast-endpoint/overview?o=7474647629856771

Verder wordt er gesplits op een datum omdat alles sequentiele data is, hierdoor kan je het niet husselen en zomaar een train/test split maken. Er zijn meerdere pogingen gedaan om een andere datum te pakken om een betere verhouding te krijgen maar dit ging lastig, daarom is er voor een bredere train/test split gekozen van ~30%.

In [0]:
# Time-based split
split_date = "2011-10-01"

ecom_train = ecom_features.filter(
    col("invoice_date") < split_date
)

ecom_test = ecom_features.filter(
    col("invoice_date") >= split_date
)

print("Train rows:", ecom_train.count())
print("Test rows:", ecom_test.count())

In [0]:
train_pd = (
    ecom_train
    .select(
        "Quantity",
        "UnitPrice",
        "year",
        "month",
        "day",
        "day_of_week",
        "revenue").toPandas())

test_pd = (
    ecom_test
    .select(
        "Quantity",
        "UnitPrice",
        "year",
        "month",
        "day",
        "day_of_week",
        "revenue").toPandas())

X_train = train_pd.drop(columns=["revenue"])
y_train = train_pd["revenue"]

X_test = test_pd.drop(columns=["revenue"])
y_test = test_pd["revenue"]

In [0]:
# Eventuele oude run afsluiten
while mlflow.active_run() is not None:
    mlflow.end_run()


with mlflow.start_run(run_name="ecommerce_sklearn_rf"):

    rf = SKLearnRF(
        n_estimators=50,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    preds = rf.predict(X_test)

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            preds
        )
    )

    mlflow.log_metric(
        "rmse",
        float(rmse)
    )

    mlflow.log_param(
        "max_depth",
        10
    )

    mlflow.log_param(
        "n_estimators",
        50
    )

    signature = infer_signature(
        X_train,
        rf.predict(X_train[:5])
    )

    mlflow.sklearn.log_model(
        sk_model=rf,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(1)
    )

    print(f"RMSE: {rmse:.4f}")

In [0]:
os.environ["MLFLOW_DFS_TMP"] = ("/Volumes/workspace/default/course_files/tmp/")

ecom_model_uri = ("runs:/ce374b1446e44aff99a1e2ae98a586f4/model")

retail_model_uri = ("runs:/e6cacef44b0644a38bda4edd835bf1a0/best_model")

loaded_ecom_model = mlflow.sklearn.load_model(ecom_model_uri)

loaded_retail_model = mlflow.spark.load_model(retail_model_uri)

## Edge model training
Nu er een cloudmodel is getraind om de verkoop te voorspellen, wordt een edge model ontwikkeld dat het aantal klanten in de winkel inschat. Dit model maakt gebruik van camerabeelden en wordt lokaal uitgevoerd op edge-apparaten voor real-time inzichten.

Er wordt hiervoor een klein CNN getraind.

In [0]:
# Images dataset verkleinen naar 64x64
images_small = np.array([
    np.array(Image.fromarray(img).resize((64, 64)))
    for img in images
])
# Labels defineren (aantal mensen in de foto)
y = labels

# Normaliseren
X_small = images_small.astype("float32") / 255.0

X_train_small, X_test_small, y_train_small, y_test_small = train_test_split(
    X_small,
    y,
    test_size=0.2,
    random_state=42
)

In [0]:
# (Tijdelijke) opslagplek voor de modellen defineren
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/course_files/tmp/"

with mlflow.start_run(run_name="CNN"):

    cnn = Sequential([

        Conv2D(16,(3,3),activation="relu",input_shape=X_train_small.shape[1:]),

        MaxPooling2D((2,2)),

        Conv2D(32,(3,3),activation="relu"),

        MaxPooling2D((2,2)),

        Flatten(),

        Dense(32,activation="relu"),

        Dense(1)
    ])

    cnn.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"])

    history = cnn.fit(
        X_train_small,
        y_train_small,
        validation_split=0.2,
        epochs=10,
        batch_size=32,
        verbose=1)

    preds = cnn.predict(X_test_small).flatten()

    mse = mean_squared_error(y_test_small,preds)
    rmse = np.sqrt(mse)

    mlflow.log_param("model_type","CNN")
    mlflow.log_param("epochs",10)
    mlflow.log_metric("rmse",rmse)
    mlflow.keras.log_model(cnn,"edge_model_cnn")
    print("CNN RMSE:", rmse)

In [0]:
#Model ophalen
model_uri = "runs:/7c984b3be2f4400b8a1d5ac18af0bd0d/edge_model_cnn"
#Model laden
edge_model_loaded = mlflow.keras.load_model(model_uri)
#Voorspellingen maken
sample = X_test_small[:5]
actual = y_test_small[:5].ravel() 

preds_edge = edge_model_loaded.predict(sample)
preds_edge = preds_edge.ravel()

compare_df = pd.DataFrame({
    "actual": actual,
    "predicted": preds_edge
})
compare_df

# Deployment
Nu de data door de pipeline heen is en alle modellen getraind zijn, is het tijd voor de deployment. Om dit te doen zijn er run id's nodig die in de log staan, deze zijn te vinden in DataBricks onder "AI/ML" -> "Experiments" of door de command "mlflow.search_runs()" te gebruiken.

### Cloud model deployment


https://dbc-f91f8233-2c2e.cloud.databricks.com/ml/endpoints/ecommerce-sklearn-endpoint/overview?o=7474647629856771

Zie bijbehorend notebook

### Edge model deployment


Hieronder wordt het edge model geconvert naar tf_lite model, wat een lichtgewicht model is. Daarna wordt deze opgeslagen in de repo zodat deze gedownload kan worden.

In [0]:

converter = tf.lite.TFLiteConverter.from_keras_model(edge_model_loaded)

tflite_model = converter.convert()

with open(
    "edge_model.tflite",
    "wb"
) as f:
    f.write(tflite_model)

print("TFLite model opgeslagen")

Hier wordt het model geladen om te laten zien dat het werkt.

In [0]:
interpreter = tf.lite.Interpreter(
    model_path="edge_model.tflite")

interpreter.allocate_tensors()

print("TFLite model succesvol geladen")

Er kunnen met het model voorspellingen worden gedaan, omdat het tflite bestand hierboven is geladen.

In [0]:
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample = X_test_small[:1].astype(np.float32)

interpreter.set_tensor(
    input_details[0]["index"],
    sample
)

interpreter.invoke()

prediction = interpreter.get_tensor(
    output_details[0]["index"]
)

print(prediction,y_test_small[0])

Het model is nu te downloaden en naar een edge device te sturen, hierna kunnen er voorspellingen gedaan worden.

# Model monitoring
Als laatste gaan we een belangrijk stuk toevoegen, namelijk het monitoren van de modellen en binnenstromende data. Het kan zo zijn dat de data veranderd en/of dat de modellen niet meer toereikend zijn. Het is belangrijk om zo snel mogelijk in te kunnen grijpen als dit gebeurt.

### Performance monitoring:

In [0]:
preds_cloud_ecom_performance = loaded_ecom_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds_cloud_ecom_performance))

print("Cloud ecom RMSE:", rmse)

In [0]:
preds_cloud_retail_performance = loaded_retail_model.transform(retail_test)

rmse = retail_evaluator.evaluate(preds_cloud_retail_performance)

print("Cloud retail RMSE:", rmse)

In [0]:
# Edge monitoring
preds_edge_performance = edge_model_loaded.predict(X_test_small)
# RMSE berekenen
rmse = np.sqrt(mean_squared_error(y_test_small, preds_edge_performance))

print("Edge RMSE:", rmse)

### Drift detection:



In [0]:
def detect_drift(model, train_data, test_data, model_name,
                 model_type="sklearn", target_column=None,
                 threshold=0.20):

    if model_type == "spark":

        train_preds = model.transform(train_data)
        baseline_mae = train_preds.withColumn("abs_error",
            abs(col("prediction") - col(target_column))).agg(avg("abs_error").alias("mae")).collect()[0]["mae"]

        test_preds = model.transform(test_data)
        current_mae = test_preds.withColumn("abs_error",
            abs(col("prediction") - col(target_column))).agg(avg("abs_error").alias("mae")).collect()[0]["mae"]

    elif model_type == "sklearn":

        X_train, y_train = train_data
        X_test, y_test = test_data

        baseline_mae = mean_absolute_error(y_train,model.predict(X_train))

        current_mae = mean_absolute_error(y_test,model.predict(X_test))

    else:
        raise ValueError(
            "model_type must be 'spark' or 'sklearn'"
        )

    increase = (current_mae - baseline_mae) / baseline_mae

    print(f"\nModel: {model_name}")
    print(f"Baseline MAE: {baseline_mae:.4f}")
    print(f"Current MAE: {current_mae:.4f}")
    print(f"Increase: {increase:.2%}")

    drift_detected = increase > threshold

    if drift_detected:
        print("Performance drift detected.")
        print("Manual review required before retraining.")
    else:
        print("No performance drift detected.")

    return drift_detected

In [0]:
ecom_drift = detect_drift(
    model=loaded_ecom_model,
    train_data=(X_train, y_train),
    test_data=(X_test, y_test),
    model_name="ecom",
    model_type="sklearn")

retail_drift = detect_drift(
    model=loaded_retail_model,
    train_data=retail_train,
    test_data=retail_test,
    target_column="sales",
    model_name="retail",
    model_type="spark")

edge_drift = detect_drift(
    model=edge_model_loaded,
    train_data=(X_train_small, y_train_small),
    test_data=(X_test_small, y_test_small),
    model_name="edge_cnn",
    model_type="sklearn")
